# Occupations Matching in Greek Inscriptions: Proof of concept

Author: Petra Hermankova
Date: 21/1/2026

This notebook searches for instances of Greek occupation terms in the GIST corpus. The code was generated with the help of Claude.ai Sonnet 4.5 as a proof of concept.

## Workflow
1. Load occupation terms, collated by Hermankova from various sources, see below
2. Generate inflectional paradigms (all cases, singular/plural)
3. Strip diacritics from all forms
4. Load GIST parquet file
5. Search inscription corpus for exact matches
6. Analyze and visualize results

## Discussion of known issues

1. The GIST inscriptions contain inscriptions from archaic period (6th century BCE) to Late Antiquity (5th century CE) and not just inscriptions from the Roman period. However, the first four centuries CE represent one of the two periods with most produced inscriptions, the other one being Classical and Hellenistic period (5th-4th century BCE).
   
2. The list of occupations are based on close reading of Classical Attic inscriptions (Harris 2002), as well as Late Antique sources (Petrikovits 1981), but are in no way exhaustive, or is evenly distributed between historical periods.
   
   *References:* 
    1. Steward, Harris and Lewis 2020: https://doi-org.ez.statsbiblioteket.dk/10.1017/9781108878135; namely chapter 4 Lewis, D. M. Labour Specialization in the Athenian Economy: Occupational Hazards; 
    2. Harris, E. M., 2002. Workshop, marketplace and household: the nature of technical specialisation in classical Athens and its inﬂuence on economy and society. In P. A. Cartledge, E. E. Cohen, and L. Foxhall, eds., Money, Labour, and Land: Approaches to the Economies of Ancient Greece. London, pp. 67–99. [article with 170 occupations from Classical Athens]
    3. Petrikovits H v. Die Spezialisierung des römischen Handwerks. Handw Vor- Frühgesch Zeit 1 Hist Rechtshistorische Beitr Untersuchungen Zur Frühgesch Gilde Ber Über Kolloquien Komm Für Altertumskunde Mittel- Nordeur Den Jahren 1977 Bis 1980. 1981; 63–132.
    4. Petrikovits H v. Die Spezialisierung des römischen Handwerks II (Spätantike). Z Für Papyrol Epigr. 1981;43: 285–306. https://www-jstor-org.ez.statsbiblioteket.dk/stable/20186140?sid=primo
    5. Waltzing JP. Étude historique sur les corporations professionnelles chez les Romains depuis les origines jusqu’à la chute de l’Empire d’Occident. Louvain: C. Peeters; 1895. Vol. 4. https://archive.org/details/tudehistoriques01waltgoog
    6. Kaše, Vojtěch, Petra Heřmánková, and Adéla Sobotková. 2022. ‘Division of Labor, Specialization and Diversity in the Ancient Roman Cities: A Quantitative Approach to Latin Epigraphy’. PLOS ONE 17 (6): e0269869. https://doi.org/10.1371/journal.pone.0269869. [Same methodology for Latin inscriptions, namely the methodological section is relevant, plus the section called “Occupations and occupational categories”.]

   
3. As we discuss in the supplementary materials, the current lemmatizers do not parse the epigraphic version of the Greek language robustly enough due to the fact that they were trained and created for literary Greek, which is different to the language of inscriptions (up to 50 % false positives for inscriptions from the Classical period). Therefore, these methods leave us with false positive matches on one hand and false negative matches on the other. We propose a similar strategy as in the case of Latin inscriptions - a Python script searching the plain text of inscriptions against the full declension paradigm. The paradigms include all case forms (nom, gen, dat, acc, voc) in both singular and plural, with every form stripped of all diacritical marks.  All diacritics are stripped from both paradigms and inscription texts for robust matching in epigraphic material. This method, however, is not without its issues and represents a substantial compromise in the case of the Greek language. 
 - First, the elimination of accents and other diacritics that may convey semantic distinctions can lead to potential inconsistencies and incorrect matching, which can be mitigated through close reading or, alternatively, by more effective parts-of-speech tagging or lemmatisation methods.
 - Second, the current list of occupations consists of terms that can have in Greek multiple meanings, that can be infered from additional context (for example, the term ``ἱππικός``, a riding trainer, identified by Harris 2002 as a self-standing occupation in Classical Athens, can, according to Liddel-Scott-Jones dictionary, have multiple meanings, either as an adjective, pertinent to horse riding or horsemanship, or an equivalent to Roman Equestrian Order. Again, without knowing the syntactic context of the word's occurrence, i.e. what role the word played in the text of an inscription, we cannot be certain whether it was used as an instance of the occupation, or it had a different meaning. We hope that the advance in the NLP methods and the application of transformer architecture and the LLM will improve in the near future the robustness of matching of the syntactic context, and thus enable distinguishing between an occupation and other meaning.

4. The HISCO codes do not fully correspond with the reality of ancient economy, as they were primarily created for medieval economies. However, they still represent the state-of-the-art and we decided to map the ancient occupations to the best of our ability. For example, the HISCO code 41110 (General wholesale trade workers) was matched to the following distinct occupations: 
   1. ἔμπορος                   (Maritime trader               ) - HISCO 41110 :   129 matches
   2. σιτοπώλης                 (Grain Dealer                  ) - HISCO 41110 :     6 matches
   3. φορτηγός                  (Maritime trader               ) - HISCO 41110 :     6 matches
   4. κριθοπώλης                (Grain Dealer                  ) - HISCO 41110 :     1 matches
   5. σησαμοπώλης               (Sesame Seller                 ) - HISCO 41110 :     1 matches
   6. σησαμοπώλις               (Sesame Seller                 ) - HISCO 41110 :     1 matches
      
5. This methodology only searches for exact matches of known nouns, excluding partial matches, occupations metioned by other morphological form, such as verb or adjective. Moreover, any noun occupations not present in the list of occupations are not matched.
   

## Setup and Configuration

In [71]:
import csv
import re
import unicodedata
from collections import defaultdict, Counter
from typing import Set, Dict, List
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

In [72]:
# ===== CONFIGURATION =====

# Input files
OCCUPATIONS_FILE = './Occupations_GREEK - Occupations_updated.tsv'
INSCRIPTIONS_FILE = '../../../data/large_data/GIST_v1-1.parquet'

# Output directory
OUTPUT_DIR = Path('output')
OUTPUT_DIR.mkdir(exist_ok=True)

# Output files
PARADIGMS_FILE = OUTPUT_DIR / 'occupations_paradigms_stripped.tsv'
RESULTS_FILE = OUTPUT_DIR / 'occupation_matches.tsv'
SUMMARY_FILE = OUTPUT_DIR / 'occupation_summary.tsv'

print("Configuration loaded")
print(f"Occupations file: {OCCUPATIONS_FILE}")
print(f"Inscriptions file: {INSCRIPTIONS_FILE}")
print(f"Output directory: {OUTPUT_DIR}")

Configuration loaded
Occupations file: ./Occupations_GREEK - Occupations_updated.tsv
Inscriptions file: ./GIST_v1-1.parquet
Output directory: output


## 1. Diacritic Stripping Function

In [73]:
def remove_diacritics(text: str) -> str:
    """
    Remove all Greek diacritical marks (accents, breathings, iota subscript).
    
    Example:
        ἁλιεύς → αλιευς
        ἀγαλματοποιός → αγαλματοποιος
    """
    # Normalize to NFD (decomposed form: base char + combining diacritics)
    nfd = unicodedata.normalize('NFD', text)
    # Filter out all combining characters (category Mn = Mark, nonspacing)
    stripped = ''.join(char for char in nfd if unicodedata.category(char) != 'Mn')
    # Normalize back to NFC (composed form)
    return unicodedata.normalize('NFC', stripped)

# Test the function
test_words = ['ἁλιεύς', 'ἀγαλματοποιός', 'Ἀθήνα', 'γεωργός']
print("Testing diacritic removal:")
for word in test_words:
    print(f"  {word:20s} → {remove_diacritics(word)}")

Testing diacritic removal:
  ἁλιεύς               → αλιευς
  ἀγαλματοποιός        → αγαλματοποιος
  Ἀθήνα                → Αθηνα
  γεωργός              → γεωργος


## 2. Greek Inflection Engine

In [74]:
class GreekInflector:
    """Generate inflectional paradigms for Ancient Greek nouns."""
    
    def detect_declension(self, nominative: str) -> str:
        """Detect declension type based on nominative ending (without diacritics)."""
        nom_stripped = remove_diacritics(nominative)
        
        if nom_stripped.endswith(('ης', 'ας', 'της')):
            return '1st_masc'
        elif nom_stripped.endswith(('α', 'η')):
            return '1st_fem'
        elif nom_stripped.endswith('ις'):
            return '1st_fem_short'
        elif nom_stripped.endswith('ος'):
            return '2nd'
        elif nom_stripped.endswith('ον'):
            return '2nd_neut'
        elif nom_stripped.endswith('ευς'):
            return '3rd_eus'
        elif nom_stripped.endswith('ων'):
            return '3rd_on'
        elif nom_stripped.endswith(('ηρ', 'τηρ')):
            return '3rd_er'
        else:
            return '3rd_consonant'
    
    def inflect_1st_masc(self, stem: str, nom_ending: str) -> Set[str]:
        """First declension masculine (ναύτης type). Returns all forms."""
        forms = set()
        
        # Singular
        forms.add(stem + nom_ending)  # nominative
        forms.add(stem + nom_ending.replace('ς', 'ου'))  # genitive
        forms.add(stem + nom_ending.replace('ς', 'ω'))   # genitive alt
        forms.add(stem + nom_ending.replace('ς', 'η'))   # dative
        forms.add(stem + nom_ending.replace('ς', 'α'))   # dative alt
        forms.add(stem + nom_ending.replace('ς', 'ν'))   # accusative
        forms.add(stem + nom_ending.replace('ς', ''))   # vocative base
        forms.add(stem + 'α')  # vocative
        
        # Plural
        forms.add(stem + 'αι')   # nom/voc
        forms.add(stem + 'ων')   # genitive
        forms.add(stem + 'αις')  # dative
        forms.add(stem + 'ας')   # accusative
        
        return forms
    
    def inflect_1st_fem(self, stem: str, nom_ending: str) -> Set[str]:
        """First declension feminine."""
        forms = set()
        
        # Singular
        forms.add(stem + nom_ending)
        if nom_ending == 'α':
            forms.update([stem + 'ας', stem + 'ης', stem + 'α', stem + 'η', 
                         stem + 'αν', stem + 'ην'])
        else:
            forms.update([stem + 'ης', stem + 'η', stem + 'ην'])
        
        # Plural
        forms.update([stem + 'αι', stem + 'ων', stem + 'αις', stem + 'ας'])
        
        return forms
    
    def inflect_2nd(self, stem: str) -> Set[str]:
        """Second declension masculine/feminine."""
        return {stem + ending for ending in 
                ['ος', 'ου', 'ω', 'ον', 'ε', 'οι', 'ων', 'οις', 'ους']}
    
    def inflect_2nd_neut(self, stem: str) -> Set[str]:
        """Second declension neuter."""
        return {stem + ending for ending in 
                ['ον', 'ου', 'ω', 'α', 'ων', 'οις']}
    
    def inflect_3rd_eus(self, stem: str) -> Set[str]:
        """Third declension -εύς type."""
        return {stem + ending for ending in 
                ['ευς', 'εως', 'ει', 'εα', 'ευ', 'εις', 'εων', 'ευσι', 'εας']}
    
    def inflect_3rd_on(self, stem: str) -> Set[str]:
        """Third declension -ων type."""
        return {stem + ending for ending in 
                ['ων', 'ονος', 'ονι', 'ονα', 'ον', 'ονες', 'ονων', 'οσι', 'ονας']}
    
    def inflect_3rd_er(self, stem: str) -> Set[str]:
        """Third declension -ηρ type."""
        return {stem + ending for ending in 
                ['ηρ', 'ορος', 'ορι', 'ορα', 'ερ', 'ορ', 'ορες', 'ορων', 'ορσι', 'ορας']}
    
    def inflect_3rd_consonant(self, nominative: str) -> Set[str]:
        """Third declension consonant stem."""
        nom_stripped = remove_diacritics(nominative)
        stem = nom_stripped[:-1] if nom_stripped.endswith('ς') else nom_stripped
        
        return {nom_stripped} | {stem + ending for ending in 
                ['ος', 'ι', 'α', 'ες', 'ων', 'σι', 'ας']}
    
    def get_stem(self, nominative: str, declension: str) -> str:
        """Extract stem from nominative."""
        nom_stripped = remove_diacritics(nominative)
        
        if declension in ['1st_masc', '1st_fem', '1st_fem_short', '2nd', '2nd_neut', '3rd_on', '3rd_er']:
            return nom_stripped[:-2]
        elif declension == '3rd_eus':
            return nom_stripped[:-3]
        
        return nom_stripped
    
    def inflect(self, nominative: str) -> Set[str]:
        """Generate all inflectional forms (stripped of diacritics)."""
        declension = self.detect_declension(nominative)
        stem = self.get_stem(nominative, declension)
        nom_stripped = remove_diacritics(nominative)
        
        if declension == '1st_masc':
            # All -ης and -ας words decline the same way
            if nom_stripped.endswith('ας'):
                forms = self.inflect_1st_masc(stem, 'ας')
            else:  # ends with -ης (including -της, -ντης, etc.)
                forms = self.inflect_1st_masc(stem, 'ης')
        elif declension in ('1st_fem', '1st_fem_short'):
            ending = 'α' if nom_stripped.endswith('α') else 'η'
            forms = self.inflect_1st_fem(stem, ending)
        elif declension == '2nd':
            forms = self.inflect_2nd(stem)
        elif declension == '2nd_neut':
            forms = self.inflect_2nd_neut(stem)
        elif declension == '3rd_eus':
            forms = self.inflect_3rd_eus(stem)
        elif declension == '3rd_on':
            forms = self.inflect_3rd_on(stem)
        elif declension == '3rd_er':
            forms = self.inflect_3rd_er(stem)
        else:
            forms = self.inflect_3rd_consonant(nominative)
        
        # Strip diacritics from all forms
        return {remove_diacritics(f) for f in forms}

print("GreekInflector class loaded")

GreekInflector class loaded


## 3. Load Occupation Terms and Generate Paradigms

In [75]:
# Load occupation terms
occupations_df = pd.read_csv(OCCUPATIONS_FILE, sep='\t')

print(f"Loaded {len(occupations_df)} occupation terms")
print(f"\nColumns: {list(occupations_df.columns)}")
print(f"\nFirst 5 occupations:")
occupations_df.head()

Loaded 331 occupation terms

Columns: ['Term', 'Translation_ENG', 'Translation_LSJ', 'Category_SHL', 'HISCO_code', 'Source', 'Latin_equivalent', 'Unnamed: 7']

First 5 occupations:


,Term,Translation_ENG,Translation_LSJ,Category_SHL,HISCO_code,Source,Latin_equivalent,Unnamed: 7
0,ἀγαλματοποιός,Statue maker,NaN,Stonework,16120,Harris 2002,NaN,NaN
1,ἀγγειουργός,Jar maker,NaN,Ceramics,89120,Harris 2002,NaN,NaN
2,ἀγκιστροπώλης,Seller of fish hooks,NaN,Tools,45190,Lewis 2020,NaN,NaN
3,αἴπολος,Goatherd,NaN,"Agriculture, Animal Husbandry",61110,Harris 2002,NaN,NaN
4,ἀκέστρια,Seamstress,NaN,Textiles,79120,Harris 2002,NaN,NaN


In [76]:
# Generate paradigms for all occupations
print("Generating inflectional paradigms...")

inflector = GreekInflector()
paradigms = []

for idx, row in occupations_df.iterrows():
    term = row['Term']
    term_stripped = remove_diacritics(term)
    all_forms = inflector.inflect(term)
    
    paradigms.append({
        'nominative_original': term,
        'nominative_stripped': term_stripped,
        'translation': row.get('Translation_ENG', row.get('Translation_ENG_SHL', '')),
        'category': row.get('Category_SHL', ''),
        'hisco_code': row.get('HISCO_code', ''),
        'all_forms': '|'.join(sorted(all_forms)),
        'form_count': len(all_forms)
    })

paradigms_df = pd.DataFrame(paradigms)

print(f"Generated paradigms for {len(paradigms_df)} occupations")
print(f"Average forms per occupation: {paradigms_df['form_count'].mean():.1f}")
print(f"Total unique forms: {sum(len(set(p['all_forms'].split('|'))) for p in paradigms)}")

# Save paradigms
paradigms_df.to_csv(PARADIGMS_FILE, sep='\t', index=False)
print(f"\nParadigms saved to: {PARADIGMS_FILE}")

paradigms_df.head()

Generating inflectional paradigms...
Generated paradigms for 331 occupations
Average forms per occupation: 10.0
Total unique forms: 3320

Paradigms saved to: output/occupations_paradigms_stripped.tsv


,nominative_original,nominative_stripped,translation,category,hisco_code,all_forms,form_count
0,ἀγαλματοποιός,αγαλματοποιος,Statue maker,Stonework,16120,αγαλματοποιε|αγαλματοποιοι|αγαλματοποιοις|αγαλματοποιον|αγαλματοποιος|αγαλματοποιου|αγαλματοποιο...,9
1,ἀγγειουργός,αγγειουργος,Jar maker,Ceramics,89120,αγγειουργε|αγγειουργοι|αγγειουργοις|αγγειουργον|αγγειουργος|αγγειουργου|αγγειουργους|αγγειουργω|...,9
2,ἀγκιστροπώλης,αγκιστροπωλης,Seller of fish hooks,Tools,45190,αγκιστροπωλα|αγκιστροπωλαι|αγκιστροπωλαις|αγκιστροπωλας|αγκιστροπωλη|αγκιστροπωληα|αγκιστροπωληη...,12
3,αἴπολος,αιπολος,Goatherd,"Agriculture, Animal Husbandry",61110,αιπολε|αιπολοι|αιπολοις|αιπολον|αιπολος|αιπολου|αιπολους|αιπολω|αιπολων,9
4,ἀκέστρια,ακεστρια,Seamstress,Textiles,79120,ακεστρα|ακεστραι|ακεστραις|ακεστραν|ακεστρας|ακεστρη|ακεστρην|ακεστρης|ακεστρων,9


### Example: View Full Paradigm for a Specific Occupation

In [77]:
# Choose an occupation to examine
example_occupation = 'ἁλιεύς'  # fisherman

example = paradigms_df[paradigms_df['nominative_original'] == example_occupation].iloc[0]
print(f"Occupation: {example['nominative_original']} ({example['translation']})")
print(f"Stripped form: {example['nominative_stripped']}")
print(f"Category: {example['category']}")
print(f"Number of forms: {example['form_count']}")
print(f"\nAll forms (stripped):")
for form in sorted(example['all_forms'].split('|')):
    print(f"  {form}")

Occupation: ἁλιεύς (Fisherman)
Stripped form: αλιευς
Category: Seafaring/Maritime
Number of forms: 9

All forms (stripped):
  αλιεα
  αλιεας
  αλιει
  αλιεις
  αλιευ
  αλιευς
  αλιευσι
  αλιεων
  αλιεως


## 4. Build Search Index

In [78]:
# Create reverse index: form -> list of occupations
print("Building search index...")

occupations_dict = {}
form_index = defaultdict(list)

for idx, row in paradigms_df.iterrows():
    nom_stripped = row['nominative_stripped']
    forms = row['all_forms'].split('|')
    
    occupations_dict[nom_stripped] = {
        'nominative_original': row['nominative_original'],
        'translation': row['translation'],
        'category': row['category'],
        'hisco_code': row['hisco_code'],
        'forms': set(forms)
    }
    
    # Build reverse index
    for form in forms:
        form_index[form].append(nom_stripped)

print(f"Index built: {len(form_index)} unique forms")
print(f"Occupations indexed: {len(occupations_dict)}")

Building search index...
Index built: 3269 unique forms
Occupations indexed: 331


## 5. Load GIST Parquet File

In [79]:
# Load inscriptions from parquet file
print(f"Loading inscriptions from {INSCRIPTIONS_FILE}...")

inscriptions_df = pd.read_parquet(INSCRIPTIONS_FILE)

print(f"Loaded {len(inscriptions_df):,} inscriptions")
print(f"\nColumns in parquet file:")
print(inscriptions_df.columns.tolist())
print(f"\nDataFrame info:")
inscriptions_df.info()

Loading inscriptions from ./GIST_v1-1.parquet...
Loaded 217,863 inscriptions

Columns in parquet file:
['PHI_ID', 'reference', 'metadata', 'lines', 'text_raw', 'text_iphi', 'clean_text_conservative', 'clean_text_interpretive_word', 'lemmata', 'raw_date', 'not_before', 'not_after', 'date_type', 'region_main_id', 'region_sub_id', 'TM_ID', 'EDH_ID', 'EDCS_ID', 'EDR_ID', 'LUPA_ID', 'ISic_ID', 'IG_ID', 'atticinscriptions_ID', 'Attalus_ID', 'TMgeo_ID', 'Longitude', 'Latitude', 'TMgeo_name', 'geometry', 'polis_context_name', 'polis_context_size', 'polis_context_fame', 'urban_context', 'urban_context_city', 'urban_context_pop_est', 'is_geotemporal']

DataFrame info:
<class 'pandas.core.frame.DataFrame'>
Index: 217863 entries, 0 to 217773
Data columns (total 36 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   PHI_ID                        217863 non-null  int64  
 1   reference                     217863 non-n

In [80]:
# Preview the data
print("First few inscriptions:")
inscriptions_df.head()

First few inscriptions:


,PHI_ID,reference,metadata,lines,text_raw,text_iphi,clean_text_conservative,clean_text_interpretive_word,lemmata,raw_date,not_before,not_after,date_type,region_main_id,region_sub_id,TM_ID,EDH_ID,EDCS_ID,EDR_ID,LUPA_ID,ISic_ID,IG_ID,atticinscriptions_ID,Attalus_ID,TMgeo_ID,Longitude,Latitude,TMgeo_name,geometry,polis_context_name,polis_context_size,polis_context_fame,urban_context,urban_context_city,urban_context_pop_est,is_geotemporal
0,1,IG I³\n1,"Att. — Ath.: Akr. — stoich. 35 — c. 510-500 a. — IG I² 1 — IG I³, Add. p.935",12.0,ἔδοχσεν το͂ι δέμοι· τ̣[ὸς ἐ Σ]αλαμ̣[ῖνι κλερόχ]ος\nοἰκε͂ν ἐᾶ Σαλαμῖνι [․․5․․]λεν [․․․7․․․ Ἀθέ]νε...,εδοχσεν τοι δεμοι τ[ος ε σ]αλαμ[ινι κλεροχ]ος οικεν εα σαλαμινι [-----]λεν [------- αθε]νεσι τελ...,ἔδοχσεν το͂ι δέμοι ταλαμος οἰκε͂ν ἐᾶ Σαλαμῖνι λεν νεσι τελε͂ν καὶ στραται τ μὲ μιο͂ν ἐὰ μὲ οἰκοὰ...,ἔδοχσεν το͂ι δέμοι τὸς ἐ Σαλαμῖνι κλερόχος οἰκε͂ν ἐᾶ Σαλαμῖνι λεν Ἀθένεσι τελε͂ν καὶ στρατεύεσθα...,"['ἔδοχσεν', 'το͂ι', 'δέμοι', 'τὸς', 'ἐ', 'Σαλαμῖνι', 'κλερόχος', 'οἰκε͂ν', 'ἐᾶ', 'Σαλαμῖνι', 'λε...",c. 510-500 a.,-511.0,-499.0,"['range', 'phase', 'ca']",1701.0,1700.0,['786251'],[],[],[],[],[],['IG I³ 1'],['1672'],[],['364'],23.723985,37.972747,Athenai,b'\x01\x01\x00\x00\x00\x9f\x9e\xc6\x17W\xb97@\x18\x1ct\xf5\x82\xfcB@',Athenai,5.0,20.87,large,Athenae,58112.0,True
1,4,IG I³\n4,Att. — stoich. 38 — 485/4 a.,56.0,[․․․․․․․․․․․․․․․․․․38․․․․․․․․․․․․․․․․․․]\n[․․․․․․․․․․․․․․․․․․38․․․․․․․․․․․․․․․․․․]\n[․․․․․․․․․․․...,[--------------------------------------] [--------------------------------------] [-------------...,δέ τις αν ἒ φρορὰν μ ντέκοντα δχμὰς τ ας ℎες πρᾶχσιν δναι π τὸς τρε͂ ἐν ἀγορᾶι ἄνρυταίον καὶ τέμ...,ἐὰν δέ τις αν ἒ φρορὰν μὲ πεντέκοντα δραχμὰς τ ας ℎ ες πρᾶχσιν δὲ ἐ͂ναι πρὸς τὸς τρε͂ς ἄρχοντας...,"['ἐάν', 'τις', 'ἐν', 'ἒ', 'φρουρά', 'μή', 'πεντήκοντα', 'δραχμή', 'τ', 'ὡς', 'ὁ', 'εἰς', 'πρᾶξις...",485/4 a.,-485.0,-484.0,['range'],1701.0,1700.0,['786254'],[],[],[],[],[],"['IG I³ 4(A)', 'IG I³ 4(B)']",['1692'],[],['364'],23.723985,37.972747,Athenai,b'\x01\x01\x00\x00\x00\x9f\x9e\xc6\x17W\xb97@\x18\x1ct\xf5\x82\xfcB@',Athenai,5.0,20.87,large,Athenae,58112.0,True
2,5,IG I³\n5,Att. — c. 500 a.,6.0,[ἔδοχσε]ν [⋮ τε͂ι βολε͂ι] ⋮ καὶ [τ]ο͂ι δέμοι ⋮ ℎό̣τ̣ε Παραιβάτες [⋮ ἐγραμμάτευε] vac.\n[προτέ]λε...,[εδοχσε]ν [ τει βολει] και [τ]οι δεμοι οτε παραιβατες [ εγραμματευε] [προτε]λεια θ[υε]ν τος ιερο...,ν καὶ ο͂ι δέμοι ℎότε Παραιβάτες λεια θν τὸς ℎιεροποιὸς Ἐλευσινίον καὶ ν ευσῖνε͂ι ℎερμε͂ι Ἐναγονί...,ἔδοχσεν τε͂ι βολε͂ι καὶ το͂ι δέμοι ℎότε Παραιβάτες ἐγραμμάτευε προτέλεια θύεν τὸς ℎιεροποιὸς Ἐλε...,"['δοκέω', 'βουλή', 'καὶ', 'ὁ', 'δῆμος', 'τε', 'παραβάτης', 'γραμματεύς', 'προτέλεια', 'πίνω', 'τ...",c. 500 a.,-505.0,-495.0,"['exact', 'phase', 'ca']",1701.0,1700.0,['786255'],[],[],[],[],[],['IG I³ 5'],['1284'],[],['10345'],23.541498,38.044135,Eleusis,b'\x01\x01\x00\x00\x009VQ\x97\x9f\x8a7@\x8f\x8f\x02:\xa6\x05C@',Eleusis,1.0,1.37,small,Eleusis,1000.0,True
3,6,IG I³\n6,Att. — stoich. 23/11 — ante 460 a.,160.0,— — — — — — — — — — — — —\n[․․․․․․15․․․․․․․] δραχ̣με͂ισ̣-\n[ι ․․․․․․14․․․․․․]τ̣ες̣ τὸς ι̣ο̣\n[․․...,------------- [---------------] δραχμεισ[ι --------------]τες τος ιο [--------------]μενος δεμο ...,δραχμε͂ιστες τὸς ιο μενος δεμο το͂ν πόλεο οκε͂ι ἀνατιθα ἐάν τι το͂ν ον ἒ ℎΟΣΑΛ ο ἒ ενορ νας ℎίνα...,δραχμε͂ισι τες τὸς ιο μενος δεμο το͂ν πόλεον δοκε͂ι ἀνατιθ α ἐάν τις το͂ν ον ἒ ℎΟΣΑΛ ο ἒ ενορ να...,"['δραχμός', 'τε', 'τὶς', 'μόνος', 'δῆμος', 'ὁ', 'παλέω', 'δοκέω', 'ἀνατί', 'α', 'ἐν', 'τις', 'ὁ'...",ante 460 a.,NaN,-461.0,['ante'],1701.0,1700.0,['786256'],[],[],[],[],[],['IG I³ 6'],['6'],[],['10345'],23.541498,38.044135,Eleusis,b'\x01\x01\x00\x00\x009VQ\x97\x9f\x8a7@\x8f\x8f\x02:\xa6\x05C@',Eleusis,1.0,1.37,small,Eleusis,1000.0,True
4,11,IG I³\n11,Att. — stoich. 48 — ante med. s. V a.,22.0,[χσυμμαχία καὶ ℎόρ]κ̣ο̣[ς] Ἀ̣[θ]ε̣ν̣α̣[ίον κα]ὶ Ἐγεσταί[ον].\n[ἔδοχσεν τε͂ι βολε͂ι καὶ τ]ο͂ι [δέ...,[χσυμμαχια και ορ]κο[ς] α[θ]ενα[ιον κα]ι εγεσται[ον]. [εδοχσεν τει βολει και τ]οι [δεμοι ------ι...,κο Ἀεναὶ Ἐγεσταί ο͂ι ἐπρυτάν

In [81]:
# Identify the text field - update this based on the columns shown above
# Common field names: 'text', 'transcription', 'content', 'diplomatic_text', etc.

# IMPORTANT: Update these after seeing the columns
INSCRIPTION_ID_FIELD = 'PHI-ID'        # ← UPDATE based on columns shown above
INSCRIPTION_TEXT_FIELD = 'clean_text_interpretive_word'    # ← UPDATE based on columns shown above

# Verify these fields exist
if INSCRIPTION_ID_FIELD not in inscriptions_df.columns:
    print(f"⚠️  Warning: '{INSCRIPTION_ID_FIELD}' not found in columns")
    print(f"Available columns: {inscriptions_df.columns.tolist()}")
else:
    print(f"✓ Will use '{INSCRIPTION_ID_FIELD}' for ID")

if INSCRIPTION_TEXT_FIELD not in inscriptions_df.columns:
    print(f"⚠️  Warning: '{INSCRIPTION_TEXT_FIELD}' not found in columns")
    print(f"Available columns: {inscriptions_df.columns.tolist()}")
else:
    print(f"✓ Will use '{INSCRIPTION_TEXT_FIELD}' for text")
    # Show sample text
    sample_text = inscriptions_df[INSCRIPTION_TEXT_FIELD].iloc[0]
    print(f"\nSample text: {sample_text[:200]}...")

⚠️  Warning: 'PHI-ID' not found in columns
Available columns: ['PHI_ID', 'reference', 'metadata', 'lines', 'text_raw', 'text_iphi', 'clean_text_conservative', 'clean_text_interpretive_word', 'lemmata', 'raw_date', 'not_before', 'not_after', 'date_type', 'region_main_id', 'region_sub_id', 'TM_ID', 'EDH_ID', 'EDCS_ID', 'EDR_ID', 'LUPA_ID', 'ISic_ID', 'IG_ID', 'atticinscriptions_ID', 'Attalus_ID', 'TMgeo_ID', 'Longitude', 'Latitude', 'TMgeo_name', 'geometry', 'polis_context_name', 'polis_context_size', 'polis_context_fame', 'urban_context', 'urban_context_city', 'urban_context_pop_est', 'is_geotemporal']
✓ Will use 'clean_text_interpretive_word' for text

Sample text: ἔδοχσεν το͂ι δέμοι τὸς ἐ Σαλαμῖνι κλερόχος οἰκε͂ν ἐᾶ Σαλαμῖνι λεν Ἀθένεσι τελε͂ν καὶ στρατεύεσθαι τὰ δ’ ἐ Σαλαμῖνι μὲ μισθο͂ν ἐὰ μὲ οἰκ ο μισθόμενο ἐὰν δὲ μισθο͂ι ἀποτίνεν τὸ μισθόμενον καὶ τὸ μισθο͂ν...


## 6. Search Function

In [82]:
def search_text_for_occupations(text: str, form_index: Dict) -> Dict[str, List[str]]:
    """
    Search a text for occupation terms.
    Returns dict: occupation_stripped -> list of matched forms
    """
    # Strip diacritics from text
    text_stripped = remove_diacritics(text)
    
    found = defaultdict(list)
    
    for form, occupations in form_index.items():
        # Word boundary search
        pattern = r'\b' + re.escape(form) + r'\b'
        if re.search(pattern, text_stripped, re.UNICODE):
            for occ in occupations:
                found[occ].append(form)
    
    return dict(found)

print("Search function loaded")

Search function loaded


### Test Search Function

In [83]:
# Test with example text
test_text = "ἁλιεὺς καὶ ναύτης Ἀπολλώνιος ἀρχιτέκτων"

print(f"Test text: {test_text}")
print(f"Stripped: {remove_diacritics(test_text)}")
print(f"\nOccupations found:")

found = search_text_for_occupations(test_text, form_index)
for occ_stripped, forms in found.items():
    occ_data = occupations_dict[occ_stripped]
    print(f"  {occ_data['nominative_original']:20s} ({occ_data['translation']:30s})")
    print(f"    Matched forms: {', '.join(forms)}")

Test text: ἁλιεὺς καὶ ναύτης Ἀπολλώνιος ἀρχιτέκτων
Stripped: αλιευς και ναυτης Απολλωνιος αρχιτεκτων

Occupations found:
  ἁλιεύς               (Fisherman                     )
    Matched forms: αλιευς
  ἀρχιτέκτων           (Architect                     )
    Matched forms: αρχιτεκτων
  ναύτης               (Sailor                        )
    Matched forms: ναυτης


## 7. Search Full Inscription Corpus

In [84]:
# Search inscriptions (from DataFrame)
print("Searching inscriptions...")
print("This may take a few minutes for large corpora...\n")

results = []
stats = defaultdict(int)
inscriptions_processed = 0
inscriptions_with_hits = 0

for i, row in inscriptions_df.iterrows():
    # Progress update
    if i % 10000 == 0 and i > 0:
        print(f"Processed {i:,} inscriptions... ({inscriptions_with_hits:,} with occupations)")
    
    inscription_id = row.get(INSCRIPTION_ID_FIELD, str(i))
    text_original = row.get(INSCRIPTION_TEXT_FIELD, '')
    
    if not text_original or pd.isna(text_original):
        continue
    
    inscriptions_processed += 1
    
    # Search for occupations
    found = search_text_for_occupations(str(text_original), form_index)
    
    if found:
        inscriptions_with_hits += 1
    
    # Record results
    for occ_stripped, matched_forms in found.items():
        stats[occ_stripped] += 1
        occ_data = occupations_dict[occ_stripped]
        
        results.append({
            'inscription_id': inscription_id,
            'occupation_original': occ_data['nominative_original'],
            'occupation_stripped': occ_stripped,
            'translation': occ_data['translation'],
            'category': occ_data['category'],
            'hisco_code': occ_data['hisco_code'],
            'matched_forms': '|'.join(matched_forms),
            'form_count': len(matched_forms)
        })

print(f"\n{'='*60}")
print(f"SEARCH COMPLETE")
print(f"{'='*60}")
print(f"Total inscriptions processed: {inscriptions_processed:,}")
print(f"Inscriptions with occupations: {inscriptions_with_hits:,} ({100*inscriptions_with_hits/inscriptions_processed:.1f}%)")
print(f"Total inscription-occupation pairs: {len(results):,}")
print(f"Unique occupations found: {len(stats)}")

Searching inscriptions...
This may take a few minutes for large corpora...

Processed 10,000 inscriptions... (892 with occupations)
Processed 20,000 inscriptions... (1,236 with occupations)
Processed 30,000 inscriptions... (1,397 with occupations)
Processed 40,000 inscriptions... (1,583 with occupations)
Processed 50,000 inscriptions... (1,829 with occupations)
Processed 60,000 inscriptions... (2,116 with occupations)
Processed 70,000 inscriptions... (2,405 with occupations)
Processed 80,000 inscriptions... (2,756 with occupations)
Processed 90,000 inscriptions... (3,381 with occupations)
Processed 100,000 inscriptions... (3,837 with occupations)
Processed 110,000 inscriptions... (4,534 with occupations)
Processed 120,000 inscriptions... (4,780 with occupations)
Processed 130,000 inscriptions... (5,203 with occupations)
Processed 140,000 inscriptions... (5,397 with occupations)
Processed 150,000 inscriptions... (5,587 with occupations)
Processed 160,000 inscriptions... (5,731 with occu

## 8. Save Results

In [85]:
# Convert to DataFrame
results_df = pd.DataFrame(results)

# Save detailed results
results_df.to_csv(RESULTS_FILE, sep='\t', index=False)
print(f"Detailed results saved to: {RESULTS_FILE}")

# Create and save summary
summary = []
for occ_stripped in sorted(stats.keys(), key=lambda x: stats[x], reverse=True):
    occ_data = occupations_dict[occ_stripped]
    summary.append({
        'occupation_original': occ_data['nominative_original'],
        'occupation_stripped': occ_stripped,
        'translation': occ_data['translation'],
        'category': occ_data['category'],
        'hisco_code': occ_data['hisco_code'],
        'count': stats[occ_stripped]
    })

summary_df = pd.DataFrame(summary)
summary_df.to_csv(SUMMARY_FILE, sep='\t', index=False)
print(f"Summary saved to: {SUMMARY_FILE}")

print(f"\nFirst 10 results:")
results_df.head(10)

Detailed results saved to: output/occupation_matches.tsv
Summary saved to: output/occupation_summary.tsv

First 10 results:


,inscription_id,occupation_original,occupation_stripped,translation,category,hisco_code,matched_forms,form_count
0,4,γραμματεύς,γραμματευς,Scribe,Writing and the Book Trade,30120,γραμματεα,1
1,12,ἀρχιτέκτων,αρχιτεκτων,Architect,Construction,2120,αρχιτεκτονα,1
2,16,ἀρχιτέκτων,αρχιτεκτων,Architect,Construction,2120,αρχιτεκτονα,1
3,17,ἀρχιτέκτων,αρχιτεκτων,Architect,Construction,2120,αρχιτεκτονες,1
4,17,ἱεροποιός,ιεροποιος,Sacrificer,Miscellaneous,14990,ιεροποιοις,1
5,18,γραμματεύς,γραμματευς,Scribe,Writing and the Book Trade,30120,γραμματευς,1
6,19,ἀρχιτέκτων,αρχιτεκτων,Architect,Construction,2120,αρχιτεκτονος,1
7,20,γραμματεύς,γραμματευς,Scribe,Writing and the Book Trade,30120,γραμματεα,1
8,22,ἀρχιτέκτων,αρχιτεκτων,Architect,Construction,2120,αρχιτεκτονι,1
9,22,γραμματεύς,γραμματευς,Scribe,Writing and the Book Trade,30120,γραμματευς,1


### Brief discussion of results

1. γραμματεύς - a scribe, but also an official (secretary, registrar) in Athens, occuring in the standard opening formula of Athenian decrees. Therefore high prevalence in Attica and other allied cities should not be surprising https://www.perseus.tufts.edu/hopper/text?doc=Perseus:text:1999.04.0057:entry=grammateu/s
2. ἱερεύς - a priest, often quoted in religious decrees and dedications all over the Greek world; the high frequency is thus not surprising.
3. ἱππικός, μουσικός - not able to determine the exact context when the term is used as an occupation, mostly due to the identical morphology with an adjective.
4. Observation - needs to be tested against comparison of HISCO codes: occupations in Greek inscriptions are more focused on the soft-skills, services, education, information production, rather than production of raw materials and sustenance, detected in Latin inscriptions. That itself does not mean the occupations of the production sectors did not exist, but the epigraphic production was not tareting them in the first place.

## 9. Analysis and Visualization

### HISCO Code Analysis

In [109]:
# Extract HISCO major and minor groups
print("HISCO CODE OVERVIEW")
print("="*80)

# Add HISCO group columns to results
results_df['hisco_major'] = results_df['hisco_code'].astype(str).str[:2]
results_df['hisco_minor'] = results_df['hisco_code'].astype(str).str[:3]

# Count occupations by HISCO code
hisco_counts = results_df['hisco_code'].value_counts()
print(f"\nTotal unique HISCO codes found: {len(hisco_counts)}")
print(f"\nTop 20 HISCO codes by frequency:")
print("="*80)
for code, count in hisco_counts.head(20).items():
    # Get occupation name for this code
    sample = results_df[results_df['hisco_code'] == code].iloc[0]
    # Convert code to string for formatting
    code_str = str(code) if pd.notna(code) else ''
    print(f"{code_str:6s}  {sample['occupation_original']:25s} {sample['translation']:35s} {count:>6,}")

HISCO CODE OVERVIEW

Total unique HISCO codes found: 57

Top 20 HISCO codes by frequency:
30120   γραμματεύς                Scribe                               2,742
14990   ἱεροποιός                 Sacrificer                           2,467
6110    ἰατρός                    Doctor                                 414
18190   ἱππικός                   Riding trainer                         406
2120    ἀρχιτέκτων                Architect                              376
98110   κελευστής                 Rowing master                          218
19190   φιλόσοφος                 Philosopher/sophist                    207
17150   ποιητής                   Maker, poet                            205
13130   διδάσκαλος                Teacher                                196
72110   χαλκεύς                   Smith                                  179
17320   μουσικός                  Musician                               151
41110   ἔμπορος                   Maritime trader              

In [110]:
# HISCO Major Group Analysis
print("\nHISCO MAJOR GROUPS (First 2 Digits)")
print("="*80)

# Define HISCO major group descriptions
hisco_major_descriptions = {
    '0': '0/1 - Professional, Technical and Related Workers',
    '1': '0/1 - Professional, Technical and Related Workers',
    '2': '2 - Administrative and Managerial Workers',
    '3': '3 - Clerical and Related Workers',
    '4': '4 - Sales Workers',
    '5': '5 - Service Workers',
    '6': '6 - Agricultural, Animal Husbandry and Forestry Workers',
    '7': '7/8/9 - Production and Related Workers, Transport Equipment Operators',
    '8': '7/8/9 - Production and Related Workers, Transport Equipment Operators',
    '9': '7/8/9 - Production and Related Workers, Transport Equipment Operators'
}

major_group_counts = results_df.groupby('hisco_major').agg({
    'inscription_id': 'count',
    'occupation_original': 'nunique'
}).rename(columns={'inscription_id': 'total_matches', 'occupation_original': 'unique_occupations'})

major_group_counts = major_group_counts.sort_values('total_matches', ascending=False)

for code, row in major_group_counts.iterrows():
    desc = hisco_major_descriptions.get(code, 'Unknown')
    print(f"{code:2s}  {desc:70s}  {row['total_matches']:>6,} matches  ({row['unique_occupations']:>3} occupations)")

print(f"\nTotal: {major_group_counts['total_matches'].sum():,} matches across {major_group_counts['unique_occupations'].sum()} occupations")


HISCO MAJOR GROUPS (First 2 Digits)
30  Unknown                                                                  2,742 matches  (  2 occupations)
14  Unknown                                                                  2,467 matches  (  3 occupations)
18  Unknown                                                                    535 matches  (  7 occupations)
61  Unknown                                                                    510 matches  (  8 occupations)
17  Unknown                                                                    435 matches  (  7 occupations)
21  Unknown                                                                    376 matches  (  1 occupations)
13  Unknown                                                                    335 matches  (  6 occupations)
98  Unknown                                                                    268 matches  ( 10 occupations)
95  Unknown                                                                    263 

In [111]:
# HISCO Minor Group Analysis - Top 20
print("\nHISCO MINOR GROUPS (First 3 Digits) - Top 20")
print("="*80)

# Common HISCO minor group descriptions
hisco_minor_descriptions = {
    '02': 'Physical and Engineering Science Professionals',
    '13': 'Teachers',
    '14': 'Religious Workers',
    '16': 'Sculptors, Painters, and Related Artists',
    '17': 'Authors, Journalists and Related Writers',
    '18': 'Athletes, Sportsmen and Related Workers',
    '19': 'Professional Workers Not Elsewhere Classified',
    '30': 'Clerical Workers',
    '41': 'Wholesale Trade Managers and Workers',
    '43': 'Technical Sales Workers',
    '45': 'Retail Trade Workers',
    '49': 'Sales Workers Not Elsewhere Classified',
    '51': 'Food and Beverage Service Workers',
    '53': 'Bakers, Pastry-Cooks and Confectionary Makers',
    '54': 'Millers and Related Workers',
    '61': 'Farm Workers and Field Crop Workers',
    '62': 'Livestock Workers',
    '64': 'Fishermen, Hunters and Related Workers',
    '71': 'Miners, Quarrymen and Related Workers',
    '72': 'Metal Processors and Workers',
    '73': 'Wood Workers',
    '75': 'Spinners, Weavers, Knitters, Dyers',
    '76': 'Tanners, Fellmongers and Pelt Dressers',
    '79': 'Tailors, Dressmakers, Sewers, Upholsterers',
    '80': 'Shoemakers and Leather Goods Makers',
    '81': 'Cabinetmakers and Related Woodworkers',
    '82': 'Stone Cutters and Carvers',
    '83': 'Blacksmiths, Toolmakers and Machine-Tool Operators',
    '88': 'Jewellery and Precious Metal Workers',
    '89': 'Glass Formers, Potters and Related Workers',
    '95': 'Builders and Construction Workers',
    '96': 'Stationary Engine and Related Equipment Operators',
    '98': 'Deck Ratings, Engine-Room Ratings, and Related Workers'
}

minor_group_counts = results_df.groupby('hisco_minor').agg({
    'inscription_id': 'count',
    'occupation_original': 'nunique'
}).rename(columns={'inscription_id': 'total_matches', 'occupation_original': 'unique_occupations'})

minor_group_counts = minor_group_counts.sort_values('total_matches', ascending=False)

for code, row in minor_group_counts.head(20).iterrows():
    desc = hisco_minor_descriptions.get(code, 'Other')
    print(f"{code:3s}  {desc:60s}  {row['total_matches']:>6,} matches  ({row['unique_occupations']:>3} occupations)")


HISCO MINOR GROUPS (First 3 Digits) - Top 20
301  Other                                                          2,742 matches  (  2 occupations)
149  Other                                                          2,467 matches  (  3 occupations)
611  Other                                                            500 matches  (  7 occupations)
181  Other                                                            406 matches  (  3 occupations)
212  Other                                                            376 matches  (  1 occupations)
171  Other                                                            271 matches  (  3 occupations)
981  Other                                                            218 matches  (  6 occupations)
191  Other                                                            207 matches  (  2 occupations)
131  Other                                                            196 matches  (  2 occupations)
721  Other                                   

In [112]:
# Create summary table of all HISCO groups in the data
print("\nCREATING HISCO CATEGORY SUMMARY TABLE")
print("="*80)

# Get unique HISCO codes with their details
hisco_summary = results_df.groupby('hisco_code').agg({
    'occupation_original': lambda x: list(x.unique()),
    'translation': lambda x: list(x.unique()),
    'category': 'first',
    'inscription_id': 'count',
    'hisco_major': 'first',
    'hisco_minor': 'first'
}).rename(columns={'inscription_id': 'count'})

hisco_summary = hisco_summary.sort_values('count', ascending=False)

# Convert to a cleaner format
hisco_summary_clean = []
for code, row in hisco_summary.iterrows():
    # Handle multiple occupations with same HISCO code
    occupations = ', '.join(row['occupation_original'][:3])  # Show first 3
    if len(row['occupation_original']) > 3:
        occupations += f" (+ {len(row['occupation_original']) - 3} more)"
    
    translations = ', '.join(row['translation'][:3])
    if len(row['translation']) > 3:
        translations += f" (+ {len(row['translation']) - 3} more)"
    
    hisco_summary_clean.append({
        'hisco_code': code,
        'hisco_major': row['hisco_major'],
        'hisco_minor': row['hisco_minor'],
        'occupations': occupations,
        'translations': translations,
        'category': row['category'],
        'count': row['count']
    })

hisco_summary_df = pd.DataFrame(hisco_summary_clean)

# Save to file
hisco_summary_file = OUTPUT_DIR / 'hisco_category_summary.tsv'
hisco_summary_df.to_csv(hisco_summary_file, sep='\t', index=False)
print(f"HISCO category summary saved to: {hisco_summary_file}")

print(f"\nFirst 20 entries:")
hisco_summary_df.head(20)


CREATING HISCO CATEGORY SUMMARY TABLE
HISCO category summary saved to: output/hisco_category_summary.tsv

First 20 entries:


,hisco_code,hisco_major,hisco_minor,occupations,translations,category,count
0,30120,30,301,"γραμματεύς, γραφεύς","Scribe, Scrivener/copyist",Writing and the Book Trade,2742
1,14990,14,149,"ἱεροποιός, ἱερεύς , χρησμολόγος","Sacrificer, Priest, sacrificer, oracle interpreter",Miscellaneous,2467
2,6110,61,611,ἰατρός,Doctor,Miscellaneous,414
3,18190,18,181,"ἱππικός, ὁπλομάχος, παιδοτρίβης","Riding trainer, Weapons trainer, Physical trainer",Education and Training,406
4,2120,21,212,ἀρχιτέκτων,Architect,Construction,376
5,98110,98,981,"κελευστής, κυβερνήτης, ναύτης (+ 3 more)","Rowing master, Helmsman, Sailor (+ 3 more)",Seafaring/Maritime,218
6,19190,19,191,"φιλόσοφος, σοφιστής",Philosopher/sophist,Education and Training,207
7,17150,17,171,"ποιητής, ῥήτωρ","Maker, poet, Orator",Literature and Scholarship,205
8,13130,13,131,"διδάσκαλος, γραμματιστής","Teacher, Elementary school teacher",Education and Training,196
9,72110,72,721,"χαλκεύς, ἀργυροκόπος, χρυσώτής (+ 6 more)","Smith, Coin minter, Gilder (+ 5 more)",Metalwork,179


In [113]:
# Detailed breakdown by major group - show all occupations
print("\nDETAILED BREAKDOWN BY HISCO MAJOR GROUP")
print("="*80)

for major_code in sorted(results_df['hisco_major'].unique()):
    major_data = results_df[results_df['hisco_major'] == major_code]
    desc = hisco_major_descriptions.get(major_code, 'Unknown')
    
    print(f"\n{'='*80}")
    print(f"HISCO Major Group {major_code}: {desc}")
    print(f"{'='*80}")
    print(f"Total matches: {len(major_data):,}")
    print(f"Unique occupations: {major_data['occupation_original'].nunique()}")
    print(f"\nTop occupations in this group:")
    
    top_in_group = major_data['occupation_original'].value_counts().head(10)
    for i, (occ, count) in enumerate(top_in_group.items(), 1):
        sample = major_data[major_data['occupation_original'] == occ].iloc[0]
        # Convert HISCO code to string
        hisco_str = str(sample['hisco_code']) if pd.notna(sample['hisco_code']) else ''
        print(f"  {i:2d}. {occ:25s} ({sample['translation']:30s}) - HISCO {hisco_str:6s}: {count:>5,} matches")


DETAILED BREAKDOWN BY HISCO MAJOR GROUP

HISCO Major Group 13: Unknown
Total matches: 335
Unique occupations: 6

Top occupations in this group:
   1. διδάσκαλος                (Teacher                       ) - HISCO 13130 :   168 matches
   2. κιθαριστής                (Kithara teacher/player        ) - HISCO 13920 :    92 matches
   3. γραμματικός               (Grammarian                    ) - HISCO 13920 :    40 matches
   4. γραμματιστής              (Elementary school teacher     ) - HISCO 13130 :    28 matches
   5. ὑποδιδάσκαλος             (Teacher’s assistant           ) - HISCO 13990 :     5 matches
   6. μανθάνων                  (Student                       ) - HISCO 13990 :     2 matches

HISCO Major Group 14: Unknown
Total matches: 2,467
Unique occupations: 3

Top occupations in this group:
   1. ἱερεύς                    (Priest, sacrificer            ) - HISCO 14990 : 1,859 matches
   2. ἱεροποιός                 (Sacrificer                    ) - HISCO 14990 :   6

In [94]:
# Top 20 most frequent occupations
print("TOP 20 MOST FREQUENT OCCUPATIONS")
print("="*80)

for i, row in summary_df.head(20).iterrows():
    print(f"{i+1:2d}. {row['occupation_original']:25s} {row['translation']:35s} {row['count']:>6,}")

TOP 20 MOST FREQUENT OCCUPATIONS
 1. γραμματεύς                Scribe                               2,714
 2. ἱερεύς                    Priest, sacrificer                   1,859
 3. ἱεροποιός                 Sacrificer                             606
 4. ἰατρός                    Doctor                                 414
 5. ἀρχιτέκτων                Architect                              376
 6. ἱππικός                   Riding trainer                         191
 7. διδάσκαλος                Teacher                                168
 8. ποιητής                   Maker, poet                            161
 9. ναύκληρος                 Shipmaster                             159
10. μουσικός                  Musician                               151
11. φιλόσοφος                 Philosopher/sophist                    140
12. ἔμπορος                   Maritime trader                        129
13. ὑπηρετής                  Architect’s assistant                  126
14. παιδοτρίβης   

In [95]:
# Category distribution
category_counts = results_df['category'].value_counts()

print("\nOCCUPATION CATEGORY DISTRIBUTION")
print("="*80)
for category, count in category_counts.items():
    print(f"{category:45s} {count:>6,}")


OCCUPATION CATEGORY DISTRIBUTION
Miscellaneous                                  2,885
Writing and the Book Trade                     2,744
Education and Training                           908
Construction                                     582
Seafaring/Maritime                               362
Literature and Scholarship                       247
Drama and Entertainment                          230
Metalwork                                        179
Textiles                                         154
Sports and Competition                           129
Agriculture, Animal Husbandry                    116
Ceramics                                         116
Woodwork                                          94
Food Preparation and Retail                       69
Stonework                                         62
Transport                                         50
Miscellaneous Retail                              44
Leatherwork and Footwear                          30
Perfumes, Ar

In [96]:
# Inscriptions with multiple occupations
multi_occ = results_df.groupby('inscription_id').size()
multi_occ_inscriptions = multi_occ[multi_occ > 1]

print(f"\nINSCRIPTIONS WITH MULTIPLE OCCUPATIONS")
print("="*80)
print(f"Total inscriptions with 2+ occupations: {len(multi_occ_inscriptions):,}")
print(f"\nDistribution:")
print(multi_occ_inscriptions.value_counts().sort_index())


INSCRIPTIONS WITH MULTIPLE OCCUPATIONS
Total inscriptions with 2+ occupations: 894

Distribution:
2     647
3     150
4      54
5      28
6       6
7       5
8       2
9       1
17      1
Name: count, dtype: int64


In [97]:
# Show some examples of inscriptions with multiple occupations
if len(multi_occ_inscriptions) > 0:
    print("\nEXAMPLES OF INSCRIPTIONS WITH MULTIPLE OCCUPATIONS")
    print("="*80)
    
    for inscription_id in multi_occ_inscriptions.head(5).index:
        matches = results_df[results_df['inscription_id'] == inscription_id]
        print(f"\nInscription: {inscription_id}")
        for idx, row in matches.iterrows():
            print(f"  - {row['occupation_original']:20s} ({row['translation']})")


EXAMPLES OF INSCRIPTIONS WITH MULTIPLE OCCUPATIONS

Inscription: 100183
  - ἀργυροκόπος          (Coin minter)
  - χρυσόχοος            (Goldsmith)

Inscription: 100217
  - γραμματεύς           (Scribe)
  - ἱερεύς               (Priest, sacrificer)

Inscription: 100325
  - φιλόσοφος            (Philosopher/sophist)
  - ἱερεύς               (Priest, sacrificer)

Inscription: 100879
  - χαλκεύς              (Smith)
  - ἱερεύς               (Priest, sacrificer)

Inscription: 101152
  - διδάσκαλος           (Teacher)
  - ὑποδιδάσκαλος        (Teacher’s assistant)


## 10. Advanced Queries

In [98]:
# Filter by specific category
category = 'Seafaring/Maritime'

maritime = results_df[results_df['category'] == category]
print(f"Occupations in '{category}' category: {len(maritime)}")
print(f"\nMost common:")
print(maritime['occupation_original'].value_counts().head(10))

Occupations in 'Seafaring/Maritime' category: 362

Most common:
occupation_original
ναύκληρος     159
ἔμπορος       129
κυβερνήτης     19
ναύτης         19
κελευστής       9
ἁλιεύς          9
πορθμεύς        6
πρῳρεύς         6
φορτηγός        6
Name: count, dtype: int64


In [99]:
# Search for specific occupation
search_occupation = 'ἁλιεύς'  # fisherman

fishermen = results_df[results_df['occupation_original'] == search_occupation]
print(f"Found {len(fishermen)} inscriptions with '{search_occupation}'")
print(f"\nFirst 10 inscriptions:")
print(fishermen[['inscription_id', 'matched_forms']].head(10))

Found 9 inscriptions with 'ἁλιεύς'

First 10 inscriptions:
     inscription_id matched_forms
4061          93305        αλιεις
5606         123177        αλιεως
5773         126112        αλιεων
5792         126815        αλιευς
6242         142069        αλιεις
6283         143825        αλιευς
7248         176260        αλιεις
8844         180452        αλιεις
8853         187170        αλιεις


In [100]:
# Occupations that appear together
# Find pairs of occupations in the same inscription
from itertools import combinations

co_occurrence = defaultdict(int)

for inscription_id in multi_occ_inscriptions.index:
    occs = results_df[results_df['inscription_id'] == inscription_id]['occupation_original'].tolist()
    for pair in combinations(sorted(set(occs)), 2):
        co_occurrence[pair] += 1

print("TOP 10 OCCUPATION PAIRS APPEARING TOGETHER")
print("="*80)
for pair, count in sorted(co_occurrence.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(f"{pair[0]:25s} + {pair[1]:25s} : {count:>4}")

TOP 10 OCCUPATION PAIRS APPEARING TOGETHER
γραμματεύς                + ἱερεύς                    :  104
γραμματεύς                + ἀρχιτέκτων                :   91
γραμματεύς                + ἱεροποιός                 :   88
γραμματεύς                + παιδοτρίβης               :   61
μουσικός                  + ἱππικός                   :   57
παιδοτρίβης               + ὁπλομάχος                 :   51
ἱερεύς                    + ἱεροποιός                 :   50
γραμματεύς                + ὁπλομάχος                 :   48
ναύκληρος                 + ἔμπορος                   :   48
ἀρχιτέκτων                + ἱεροποιός                 :   47


## 11. Export Summary Statistics

In [101]:
# Create comprehensive summary statistics
summary_stats = {
    'total_inscriptions_in_corpus': len(inscriptions_df),
    'total_inscriptions_processed': inscriptions_processed,
    'inscriptions_with_occupations': inscriptions_with_hits,
    'percentage_with_occupations': 100 * inscriptions_with_hits / inscriptions_processed,
    'total_matches': len(results_df),
    'unique_occupations_found': len(stats),
    'total_occupations_searched': len(occupations_dict),
    'inscriptions_with_multiple_occupations': len(multi_occ_inscriptions),
    'max_occupations_per_inscription': multi_occ.max() if len(multi_occ) > 0 else 0
}

print("SUMMARY STATISTICS")
print("="*60)
for key, value in summary_stats.items():
    if isinstance(value, float):
        print(f"{key:40s}: {value:>10.2f}")
    else:
        print(f"{key:40s}: {value:>10,}")

# Save to file
stats_file = OUTPUT_DIR / 'summary_statistics.txt'
with open(stats_file, 'w', encoding='utf-8') as f:
    f.write("GREEK OCCUPATION SEARCH - SUMMARY STATISTICS\n")
    f.write("="*60 + "\n\n")
    for key, value in summary_stats.items():
        if isinstance(value, float):
            f.write(f"{key:40s}: {value:>10.2f}\n")
        else:
            f.write(f"{key:40s}: {value:>10,}\n")

print(f"\nStatistics saved to: {stats_file}")

SUMMARY STATISTICS
total_inscriptions_in_corpus            :    217,863
total_inscriptions_processed            :    216,240
inscriptions_with_occupations           :      7,751
percentage_with_occupations             :       3.58
total_matches                           :      9,034
unique_occupations_found                :        150
total_occupations_searched              :        331
inscriptions_with_multiple_occupations  :        894
max_occupations_per_inscription         :         17

Statistics saved to: output/summary_statistics.txt


## Summary

This notebook:
1. ✓ Loaded 297 Greek occupation terms
2. ✓ Generated inflectional paradigms for all forms
3. ✓ Stripped diacritics for robust matching
4. ✓ Loaded GIST parquet file
5. ✓ Searched the inscription corpus
6. ✓ Generated detailed results and summaries

### Output Files:
- **Paradigms**: `occupations_paradigms_stripped.tsv`
- **Detailed results**: `occupation_matches.tsv`
- **Summary**: `occupation_summary.tsv`
- **Statistics**: `summary_statistics.txt`

All files are in: `output/`